# Interactive launch-site maps

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Site markers and launch outcomes
This separate geographic dataset contains 56 launches. Marker clusters preserve coincident records at the launch pad. Green means Class=1 and red means Class=0.

In [2]:
import folium
from folium.plugins import MarkerCluster,MousePosition
geo=pd.read_csv(DATA/'spacex_launch_geo.csv')
sites=geo.groupby('Launch Site')[['Lat','Long']].first().reset_index()
display(sites)
m=folium.Map(location=[35,-96],zoom_start=4,tiles='OpenStreetMap')
for _,r in sites.iterrows():
    folium.CircleMarker([r.Lat,r.Long],radius=10,color='#176B9A',fill=True,popup=r['Launch Site'],tooltip=r['Launch Site']).add_to(m)
m.save(ROOT/'maps'/'launch_sites.html')
cluster=MarkerCluster(name='Launch outcomes').add_to(m)
for _,r in geo.iterrows():
    folium.Marker([r.Lat,r.Long],popup=f"{r['Launch Site']} | Flight {r['Flight Number']} | Class {r['class']}",icon=folium.Icon(color='green' if r['class']==1 else 'red',icon='info-sign')).add_to(cluster)
MousePosition().add_to(m);folium.LayerControl().add_to(m)
m.save(ROOT/'maps'/'launch_outcomes.html')
print('Sites:',len(sites),'Launches:',len(geo))
m

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Sites: 4 Launches: 56


## Proximity to coastline and road
Points selected manually on the OpenStreetMap display on 7 November 2026: coastline (28.5624, -80.5677) and Samuel C. Phillips Parkway (28.5624, -80.5707). Distances use the haversine formula. These are approximate straight-line distances to selected points, not road travel distances or surveyed nearest-feature distances. The old coastline coordinate in the course instructions appears inland on the current basemap, so the visually selected coordinate is used here.

In [3]:
from math import radians,sin,cos,asin,sqrt
def distance_km(a,b):
    la1,lo1,la2,lo2=map(radians,[*a,*b]);return 6373*2*asin(sqrt(sin((la2-la1)/2)**2+cos(la1)*cos(la2)*sin((lo2-lo1)/2)**2))
row=sites[sites['Launch Site']=='CCAFS LC-40'].iloc[0]
site=[row.Lat,row.Long]
references={'Coastline':[28.5624,-80.5677],'Samuel C. Phillips Parkway':[28.5624,-80.5707]}
near=folium.Map(location=site,zoom_start=15)
folium.Marker(site,tooltip='CCAFS LC-40').add_to(near)
proximities=[]
for label,point in references.items():
    km=distance_km(site,point);proximities.append({'Feature':label,'Latitude':point[0],'Longitude':point[1],'Distance_km':km})
    folium.Marker(point,tooltip=f'{label}: {km:.3f} km',popup=f'{label}: {km:.3f} km',icon=folium.Icon(color='green')).add_to(near)
    folium.PolyLine([site,point],color='blue',tooltip=f'{km:.3f} km').add_to(near)
MousePosition().add_to(near)
near.save(ROOT/'maps'/'proximity.html')
(OUT/'map_results.json').write_text(json.dumps({'proximities':proximities,'sites':sites.to_dict('records'),'records':len(geo)},indent=2))
display(pd.DataFrame(proximities))
near

,Feature,Latitude,Longitude,Distance_km
0,Coastline,28.5624,-80.5677,0.943432
1,Samuel C. Phillips Parkway,28.5624,-80.5707,0.650382
